# 05 — Generate inventory

**Business objective:** stock-outs are a plausible contributing cause to a
sales drop, so we need periodic inventory snapshots to let later analysis
rule this in or out for Sydney specifically.

**What we're generating:** weekly stock-level snapshots per store/product
combination, with a random subset of products sampled per store (not every
store carries every product).

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np

rng = np.random.default_rng(cfg.SEED + 4)

stores = pd.read_csv(cfg.DATA_DIR / "stores.csv")
products = pd.read_csv(cfg.DATA_DIR / "products.csv")

## Generation logic

Each store stocks 70% of the catalog, sampled independently. Stock levels fluctuate weekly around a per-product baseline, with no engineered anomaly here — this table exists so validation can *rule out* stock-outs as the Sydney cause, which is itself a useful finding.

In [2]:
snapshot_dates = pd.date_range("2026-01-05", "2026-06-29", freq="W")

rows = []
inv_id = 1
for store in stores.itertuples():
    carried = products.sample(frac=0.7, random_state=cfg.SEED + store.store_id)["product_id"].tolist()
    baseline = {pid: rng.integers(20, 150) for pid in carried}
    for d in snapshot_dates:
        for pid in carried:
            level = max(0, int(rng.normal(baseline[pid], baseline[pid] * 0.15)))
            rows.append({
                "inventory_id": inv_id,
                "store_id": store.store_id,
                "product_id": pid,
                "snapshot_date": d,
                "stock_level": level,
                "reorder_level": int(baseline[pid] * 0.2),
            })
            inv_id += 1

inventory = pd.DataFrame(rows)
print(f"Generated {len(inventory):,} inventory snapshot rows")
inventory.head()

Generated 28,000 inventory snapshot rows


,inventory_id,store_id,product_id,snapshot_date,stock_level,reorder_level
0,1,1,57,2026-01-11,81,17
1,2,1,38,2026-01-11,121,27
2,3,1,68,2026-01-11,52,9
3,4,1,80,2026-01-11,34,6
4,5,1,81,2026-01-11,87,18


## Sample output & distribution check

In [3]:
print(inventory["stock_level"].describe())
print()
print("Snapshots per store:")
print(inventory.groupby("store_id").size())

count    28000.000000
mean        83.777571
std         38.660642
min         12.000000
25%         51.000000
50%         82.000000
75%        113.000000
max        206.000000
Name: stock_level, dtype: float64

Snapshots per store:
store_id
1    3500
2    3500
3    3500
4    3500
5    3500
6    3500
7    3500
8    3500
dtype: int64


## Validation

In [4]:
assert inventory["inventory_id"].is_unique
assert inventory["store_id"].isin(stores["store_id"]).all()
assert inventory["product_id"].isin(products["product_id"]).all()
assert (inventory["stock_level"] >= 0).all()
assert inventory.isnull().sum().sum() == 0
print("All validation checks passed.")

All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "inventory.csv"
inventory.to_csv(out_path, index=False)
print(f"Wrote {len(inventory):,} rows to {out_path}")

Wrote 28,000 rows to /home/claude/project/eaida/data/raw/inventory.csv


## Summary

Generated weekly stock snapshots for 70% of the catalog per store, with no
engineered anomaly — deliberately, so this table can help *rule out*
stock-outs as a cause of the Sydney drop later. Saved to
`data/raw/inventory.csv`.